In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
import numpy as np
from PIL import Image
import os
from pathlib import Path

In [2]:

# Dataset 
class Mini9Dataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = Path(root_dir)
        self.transform = transform
        self.classes = ['airplane', 'automobile', 'bird', 'cat', 'deer', 
                        'dog', 'horse', 'ship', 'truck']
        self.class_to_idx = {cls: i for i, cls in enumerate(self.classes)}
        
        
        self.images = []
        self.labels = []
        for cls in self.classes:
            cls_dir = self.root_dir / cls
            if cls_dir.exists():
                for img_path in cls_dir.glob('*.jpg'):
                    self.images.append(str(img_path))
                    self.labels.append(self.class_to_idx[cls])
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        img = Image.open(self.images[idx]).convert('RGB')
        label = self.labels[idx]
        if self.transform:
            img = self.transform(img)
        return img, label



In [3]:

# MixUp & CutMix
def mixup_data(x, y, alpha=1.0):
    """MixUp augmentation"""
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1
    
    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(x.device)
    
    mixed_x = lam * x + (1 - lam) * x[index]
    y_a, y_b = y, y[index]
    
    return mixed_x, y_a, y_b, lam


def cutmix_data(x, y, alpha=1.0):
    """CutMix augmentation"""
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1
    
    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(x.device)
    
    # Generate random box
    _, _, H, W = x.size()
    cut_rat = np.sqrt(1. - lam)
    cut_w = int(W * cut_rat)
    cut_h = int(H * cut_rat)
    
    cx = np.random.randint(W)
    cy = np.random.randint(H)
    
    bbx1 = np.clip(cx - cut_w // 2, 0, W)
    bby1 = np.clip(cy - cut_h // 2, 0, H)
    bbx2 = np.clip(cx + cut_w // 2, 0, W)
    bby2 = np.clip(cy + cut_h // 2, 0, H)
    
    # Mix images
    x[:, :, bby1:bby2, bbx1:bbx2] = x[index, :, bby1:bby2, bbx1:bbx2]
    
    # Adjust lambda based on actual box size
    lam = 1 - ((bbx2 - bbx1) * (bby2 - bby1) / (W * H))
    y_a, y_b = y, y[index]
    
    return x, y_a, y_b, lam


def mixup_criterion(criterion, pred, y_a, y_b, lam):
    """Combined loss for MixUp/CutMix"""
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)


In [4]:

#  Label Smoothing Loss 
class LabelSmoothingCrossEntropy(nn.Module):
    """CrossEntropy with Label Smoothing"""
    def __init__(self, smoothing=0.1):
        super().__init__()
        self.smoothing = smoothing
        self.confidence = 1.0 - smoothing
    
    def forward(self, pred, target):
        pred = pred.log_softmax(dim=-1)
        
        with torch.no_grad():
            true_dist = torch.zeros_like(pred)
            true_dist.fill_(self.smoothing / (pred.size(-1) - 1))
            true_dist.scatter_(1, target.data.unsqueeze(1), self.confidence)
        
        return torch.mean(torch.sum(-true_dist * pred, dim=-1))


In [5]:

# ResNet Architecture 
class ResidualBlock(nn.Module):
    """Basic Residual Block"""
    def __init__(self, in_channels, out_channels, stride=1, dropout=0.0):
        super().__init__()
        
        # Main path
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.dropout = nn.Dropout2d(dropout) if dropout > 0 else None
        
        # Shortcut
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )
    
    def forward(self, x):
        residual = x
        
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        
        if self.dropout:
            out = self.dropout(out)
        
        out += self.shortcut(residual)
        out = self.relu(out)
        
        return out


class Mini9ResNet(nn.Module):
    """ResNet-Inspired for Mini9"""
    def __init__(self, num_classes=9):
        super().__init__()
        
        # Initial conv
        self.conv1 = nn.Conv2d(3, 64, 3, 1, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        
        # Residual layers
        self.layer1 = self._make_layer(64, 64, 2, stride=1, dropout=0.1)
        self.layer2 = self._make_layer(64, 128, 2, stride=2, dropout=0.2)
        self.layer3 = self._make_layer(128, 256, 2, stride=2, dropout=0.3)
        self.layer4 = self._make_layer(256, 512, 2, stride=2, dropout=0.4)
        
        # Global average pooling
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        
        # Classifier
        self.fc = nn.Linear(512, num_classes)
        
        # Initialize weights
        self._initialize_weights()
    
    def _make_layer(self, in_channels, out_channels, num_blocks, stride, dropout):
        layers = []
        # First block (may downsample)
        layers.append(ResidualBlock(in_channels, out_channels, stride, dropout))
        # Remaining blocks
        for _ in range(1, num_blocks):
            layers.append(ResidualBlock(out_channels, out_channels, 1, dropout))
        return nn.Sequential(*layers)
    
    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.constant_(m.bias, 0)
    
    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        
        return x

In [6]:

#  Training Functions
def train_epoch(model, loader, criterion, optimizer, device, use_mixup=True, 
                mixup_alpha=1.0, cutmix_prob=0.5):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        # Apply MixUp or CutMix randomly
        if use_mixup and np.random.rand() < 1.0:
            if np.random.rand() < cutmix_prob:
                # CutMix
                inputs, labels_a, labels_b, lam = cutmix_data(inputs, labels, mixup_alpha)
            else:
                # MixUp
                inputs, labels_a, labels_b, lam = mixup_data(inputs, labels, mixup_alpha)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = mixup_criterion(criterion, outputs, labels_a, labels_b, lam)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            # For accuracy calculation with mixed labels
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += (lam * predicted.eq(labels_a).sum().float() + 
                       (1 - lam) * predicted.eq(labels_b).sum().float()).item()
        else:
            # Normal training
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    
    return running_loss / len(loader), 100. * correct / total


def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    
    return running_loss / len(loader), 100. * correct / total



In [ ]:

# Main Training 
def main():
    # Device setup (M4 GPU)
    device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
    print(f"Using device: {device}")
    
    # Hyperparameters
    BATCH_SIZE = 128
    EPOCHS = 150
    LR = 0.1
    WEIGHT_DECAY = 5e-4
    LABEL_SMOOTHING = 0.1
    MIXUP_ALPHA = 1.0
    CUTMIX_PROB = 0.5
    
    # Data transforms
    mean = [0.52461963, 0.55828897, 0.58782098]
    std = [0.18552486, 0.18374406, 0.19512308]
    
    train_transform = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ])
    
    val_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ])
    
    # Datasets & Dataloaders
    train_dataset = Mini9Dataset('train', transform=train_transform)
    val_dataset = Mini9Dataset('val', transform=val_transform)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, 
                             shuffle=True, num_workers=0, pin_memory=False)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, 
                           shuffle=False, num_workers=0, pin_memory=False)
    
    print(f"Train size: {len(train_dataset)}, Val size: {len(val_dataset)}")
    
    # Model
    model = Mini9ResNet(num_classes=9).to(device)
    print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
    
    # Loss with Label Smoothing
    criterion = LabelSmoothingCrossEntropy(smoothing=LABEL_SMOOTHING)
    
    # Optimizer
    optimizer = optim.SGD(model.parameters(), lr=LR, momentum=0.9, 
                         weight_decay=WEIGHT_DECAY, nesterov=True)
    
    # Cosine Annealing Scheduler
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
    
    # Training loop
    best_val_acc = 0
    patience_counter = 0
    patience = 25
    
    print("\n" + "="*60)
    print("Training Started with:")
    print(f"  - MixUp/CutMix (alpha={MIXUP_ALPHA}, cutmix_prob={CUTMIX_PROB})")
    print(f"  - Label Smoothing (smoothing={LABEL_SMOOTHING})")
    print(f"  - Cosine Annealing LR (T_max={EPOCHS})")
    print("="*60 + "\n")
    
    for epoch in range(EPOCHS):
        train_loss, train_acc = train_epoch(model, train_loader, criterion, 
                                           optimizer, device, 
                                           use_mixup=True, 
                                           mixup_alpha=MIXUP_ALPHA,
                                           cutmix_prob=CUTMIX_PROB)
        val_loss, val_acc = validate(model, val_loader, criterion, device)
        
        # Step scheduler
        scheduler.step()
        
        print(f"Epoch {epoch+1}/{EPOCHS}")
        print(f"  Train - Loss: {train_loss:.4f}, Acc: {train_acc:.2f}%")
        print(f"  Val   - Loss: {val_loss:.4f}, Acc: {val_acc:.2f}%")
        print(f"  LR: {optimizer.param_groups[0]['lr']:.6f}")
        
        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), 'best_model.pth')
            print(f"  ✓ New best model saved! Acc: {val_acc:.2f}%")
            patience_counter = 0
        else:
            patience_counter += 1
        
        # Early stopping
        if patience_counter >= patience:
            print(f"\nEarly stopping at epoch {epoch+1}")
            break
        
        print()
    
    print(f"\n{'='*60}")
    print(f"Training Complete!")
    print(f"Best Validation Accuracy: {best_val_acc:.2f}%")
    print(f"{'='*60}")


if __name__ == "__main__":
    main()

Using device: mps
Train size: 45000, Val size: 9000
Model parameters: 11,173,449

Training Started with:
  - MixUp/CutMix (alpha=1.0, cutmix_prob=0.5)
  - Label Smoothing (smoothing=0.1)
  - Cosine Annealing LR (T_max=150)

Epoch 1/150
  Train - Loss: 2.5175, Acc: 12.38%
  Val   - Loss: 2.1808, Acc: 15.18%
  LR: 0.099989
  ✓ New best model saved! Acc: 15.18%

Epoch 2/150
  Train - Loss: 2.1422, Acc: 17.46%
  Val   - Loss: 1.9908, Acc: 24.57%
  LR: 0.099956
  ✓ New best model saved! Acc: 24.57%

Epoch 3/150
  Train - Loss: 2.0302, Acc: 23.77%
  Val   - Loss: 1.8423, Acc: 34.70%
  LR: 0.099901
  ✓ New best model saved! Acc: 34.70%

Epoch 4/150
  Train - Loss: 1.9570, Acc: 29.01%
  Val   - Loss: 1.6865, Acc: 41.96%
  LR: 0.099825
  ✓ New best model saved! Acc: 41.96%

Epoch 5/150
  Train - Loss: 1.8898, Acc: 33.45%
  Val   - Loss: 1.5888, Acc: 48.67%
  LR: 0.099726
  ✓ New best model saved! Acc: 48.67%

Epoch 6/150
  Train - Loss: 1.8486, Acc: 36.35%
  Val   - Loss: 1.5261, Acc: 52.20%
  

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report
import pandas as pd
import os





# Collect model predictions on the validation set

def collect_predictions(model, loader, device):
    model.eval()
    all_labels, all_preds = [], []
    
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            _, preds = model(x).max(1)
            all_labels.extend(y.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
    
    return all_labels, all_preds



# Main evaluation: confusion matrix + per-class metrics

def analyze_model(model_path="best_model.pth", val_dir="val"):

    device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
    print(f"Device: {device}")

    # Load model weights
    model = Mini9ResNet(num_classes=9).to(device)
    if not os.path.exists(model_path):
        print("Model file not found.")
        return
    model.load_state_dict(torch.load(model_path, map_location=device))

    # Same normalization used during training
    mean = [0.52461963, 0.55828897, 0.58782098]
    std  = [0.18552486, 0.18374406, 0.19512308]

    val_tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ])

    # Validation loader
    val_set = Mini9Dataset(val_dir, transform=val_tf)
    val_loader = DataLoader(val_set, batch_size=256, shuffle=False)

    print(f"Validation samples: {len(val_set)}")

    # Run inference
    labels, preds = collect_predictions(model, val_loader, device)

    # Compute metrics
    classes = val_set.classes
    cm = confusion_matrix(labels, preds)
    report = classification_report(labels, preds, target_names=classes, output_dict=True)

    print("\n============================================")
    print(f"Overall Accuracy: {report['accuracy']*100:.2f}%")
    print("============================================\n")

    print("Confusion Matrix:")
    print(pd.DataFrame(cm, index=classes, columns=classes))

    print("\nPer-Class Metrics:")
    df = pd.DataFrame({
        cls: {
            "Support": report[cls]["support"],
            "Accuracy": f"{report[cls]['recall']*100:.2f}%",
            "Precision": report[cls]["precision"],
            "F1": report[cls]["f1-score"]
        }
        for cls in classes
    }).T

    print(df)
    print("\n============================================")


if __name__ == "__main__":
    analyze_model()


Device: mps
Validation samples: 9000

Overall Accuracy: 92.93%

Confusion Matrix:
            airplane  automobile  bird  cat  deer  dog  horse  ship  truck
airplane         942           2    16    7     5    0      1    20      7
automobile         1         972     2    3     0    0      1     3     18
bird              14           0   917   25    24   16      1     3      0
cat                9           2    31  845    20   81      4     5      3
deer               2           0    27   17   926   17      9     2      0
dog                3           1    15   71    17  887      6     0      0
horse              6           0     7    9     9   15    952     0      2
ship              16           4     4    3     0    0      0   968      5
truck              4          25     2    4     0    1      1     8    955

Per-Class Metrics:
           Support Accuracy Precision        F1
airplane    1000.0   94.20%  0.944835  0.943415
automobile  1000.0   97.20%  0.966203  0.969093
bird